# Agentes de Código: Ejecución Segura y Sandboxing

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LegalIntermediaSL/InteligenciaArtificial/blob/main/tutoriales/notebooks/agentes-avanzados/09-agentes-codigo.ipynb)

Cómo construir agentes que generen y ejecuten código de forma segura con sandboxing.

In [ ]:
!pip install anthropic pandas matplotlib -q

In [ ]:
import anthropic
import io
import sys
import traceback
import ast

client = anthropic.Anthropic()

## 1. Ejecución segura con exec() y captura de output

In [ ]:
IMPORTS_PERMITIDOS = {'pandas', 'numpy', 'matplotlib', 'math', 'statistics', 'json', 'datetime', 're', 'collections'}
OPERACIONES_PELIGROSAS = ['import os', 'import sys', 'import subprocess', '__import__', 'eval(', 'exec(', 'open(']

def es_codigo_seguro(codigo: str) -> tuple[bool, str]:
    """Valida que el código no contenga operaciones peligrosas."""
    for op in OPERACIONES_PELIGROSAS:
        if op in codigo:
            return False, f'Operación no permitida: {op}'
    try:
        ast.parse(codigo)
    except SyntaxError as e:
        return False, f'Error de sintaxis: {e}'
    return True, 'OK'

def ejecutar_codigo_seguro(codigo: str, timeout: int = 10) -> dict:
    """Ejecuta código Python con sandbox básico y captura de output."""
    seguro, motivo = es_codigo_seguro(codigo)
    if not seguro:
        return {'exito': False, 'error': f'Código bloqueado: {motivo}', 'stdout': '', 'stderr': ''}

    stdout_capture = io.StringIO()
    stderr_capture = io.StringIO()

    namespace = {}
    try:
        sys.stdout = stdout_capture
        sys.stderr = stderr_capture
        exec(codigo, namespace)  # noqa: S102
        sys.stdout = sys.__stdout__
        sys.stderr = sys.__stderr__
        return {'exito': True, 'stdout': stdout_capture.getvalue(), 'stderr': '', 'error': None}
    except Exception as e:
        sys.stdout = sys.__stdout__
        sys.stderr = sys.__stderr__
        return {'exito': False, 'stdout': stdout_capture.getvalue(), 'stderr': traceback.format_exc(), 'error': str(e)}

# Tests
casos = [
    'print(sum(range(1, 11)))',
    'import os; os.system("ls")',  # Bloqueado
    'resultado = [x**2 for x in range(5)]\nprint(resultado)',
]

for codigo in casos:
    r = ejecutar_codigo_seguro(codigo)
    estado = '✅' if r['exito'] else '🚫'
    print(f'{estado} {codigo[:50]}')
    if r['stdout']:
        print(f'   Output: {r["stdout"].strip()}')
    if r['error']:
        print(f'   Error: {r["error"]}')

## 2. Agente analista de datos

In [ ]:
import pandas as pd

# Dataset de ejemplo
df_ventas = pd.DataFrame({
    'mes': ['Ene', 'Feb', 'Mar', 'Abr', 'May', 'Jun'],
    'ventas': [12500, 15300, 14200, 18900, 21000, 19500],
    'costes': [8000, 9200, 8800, 11500, 13000, 12100],
})
df_ventas['beneficio'] = df_ventas['ventas'] - df_ventas['costes']

def agente_analista(pregunta: str, dataframe: pd.DataFrame) -> str:
    """Agente que genera y ejecuta código Pandas para responder preguntas."""
    schema = f"Columnas: {list(dataframe.columns)}. Primeras filas:\n{dataframe.head(3).to_string()}"

    # Pedir a Claude que genere el código
    response = client.messages.create(
        model='claude-sonnet-4-6',
        max_tokens=512,
        system='Genera código Python que usa una variable `df` (ya definida) para responder. Solo código, sin explicaciones.',
        messages=[{
            'role': 'user',
            'content': f'DataFrame schema:\n{schema}\n\nPregunta: {pregunta}\n\nGenera código Python con `print()` para mostrar el resultado.'
        }],
    )

    codigo = response.content[0].text.strip().strip('`').replace('python\n', '')

    # Ejecutar con el dataframe inyectado
    namespace = {'df': dataframe, 'pd': pd}
    stdout_capture = io.StringIO()
    try:
        sys.stdout = stdout_capture
        exec(codigo, namespace)  # noqa: S102
        sys.stdout = sys.__stdout__
        return stdout_capture.getvalue().strip()
    except Exception as e:
        sys.stdout = sys.__stdout__
        return f'Error: {e}'

# Hacer preguntas al agente
preguntas = [
    '¿Cuál es el mes con mayor beneficio?',
    '¿Cuál es el beneficio medio mensual?',
    '¿Cuánto suman las ventas totales?',
]

for pregunta in preguntas:
    respuesta = agente_analista(pregunta, df_ventas)
    print(f'Q: {pregunta}')
    print(f'A: {respuesta}\n')

## 3. Generador de tests unitarios

In [ ]:
def generar_tests(codigo_funcion: str) -> str:
    """Genera tests unitarios para una función Python."""
    response = client.messages.create(
        model='claude-sonnet-4-6',
        max_tokens=1024,
        messages=[{
            'role': 'user',
            'content': f"""Genera tests unitarios con pytest para esta función.
Incluye: casos normales, edge cases, y casos de error.
Solo el código de tests, sin explicaciones.

```python
{codigo_funcion}
```"""
        }],
    )
    return response.content[0].text

funcion_ejemplo = '''
def calcular_descuento(precio: float, porcentaje: float) -> float:
    if precio < 0 or porcentaje < 0 or porcentaje > 100:
        raise ValueError('Valores inválidos')
    return precio * (1 - porcentaje / 100)
'''

tests = generar_tests(funcion_ejemplo)
print('Tests generados:')
print(tests[:800])